# Darwin's Gambit - Cloud Evolution (Google Colab)

Runs the GA / neural-net training in-notebook so progress streams live.

CPU only - do NOT pick a GPU runtime. Runtime > Change runtime type > CPU.

IMPORTANT: upload the LATEST training.zip (it contains the residual NN fix).


### 1. Install dependencies


In [ ]:
!pip -q install chess numpy


### 2. Upload the latest training.zip (from your D:\chessai folder)


In [ ]:
from google.colab import files
import zipfile, os, glob, sys
up = files.upload()                       # choose training.zip
with zipfile.ZipFile(next(iter(up))) as z: z.extractall('.')
TRAIN = os.path.dirname(glob.glob('**/evolve.py', recursive=True)[0])
if TRAIN not in sys.path: sys.path.insert(0, TRAIN)
print('training dir =', TRAIN)


### 3. Smoke test (about 1-2 min): a line per generation should appear


In [ ]:
from ga import evolve, EvolutionConfig

def report(rec):
    print(f"gen {rec.index:>2}: fitness {rec.best_fitness:5.1f} | win-rate vs baseline {rec.benchmark_winrate*100:5.1f}%", flush=True)

print('smoke run...', flush=True)
evolve(EvolutionConfig(population=8, generations=4, depth=2, max_plies=60,
                       benchmark_openings=8, processes=2, seed=0, genome_kind='nn'), on_generation=report)
print('smoke done', flush=True)


### 4. The real run (a few hours). Residual NN starts near baseline and climbs.
More benchmark_openings = a smoother, more reliable champion. Bump population/
generations for more strength; set processes=1 if a pool ever hangs.


In [ ]:
import os, json
from ga import evolve, EvolutionConfig

def report(rec):
    print(f"gen {rec.index:>2}: fitness {rec.best_fitness:5.1f} | win-rate vs baseline {rec.benchmark_winrate*100:5.1f}%", flush=True)

cfg = EvolutionConfig(population=16, generations=30, depth=2, max_plies=100,
                      benchmark_openings=16, processes=2, seed=1, genome_kind='nn')
print('evolving the neural net (residual)...', flush=True)
result = evolve(cfg, on_generation=report)

out = os.path.join(TRAIN, 'nn_weights.json')
json.dump(result.best.genome.to_dict(), open(out, 'w'))
print('DONE - saved', out, flush=True)
print(f'final-gen win-rate vs baseline: {result.run.generations[-1].benchmark_winrate*100:.1f}%', flush=True)


### 5. Download the trained net
Put nn_weights.json into your local web/ folder (replace the old one); the
'Neural Net (evolved)' opponent then uses it.


In [ ]:
from google.colab import files
files.download(os.path.join(TRAIN, 'nn_weights.json'))
